<a href="https://colab.research.google.com/github/kaioc89/Topicos_Avancados_2026-1_Equipe_JUD_4_atividade1_Kaio/blob/main/Atividade1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Instalação de dependências

In [ ]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:
pip install transformers accelerate bitsandbytes datasets bert_score sentence-transformers

# Inicialização

In [ ]:
import torch
import sys
import argparse
import os
import re
import json
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from google.colab import userdata

# Check if CUDA is visible
print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# REMOVIDO TOKEN EXPOSTO: Agora utiliza Secrets do Colab
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("✅ HF_TOKEN carregado com sucesso dos Secrets.")
except Exception:
    HF_TOKEN = None
    print("⚠️ Aviso: HF_TOKEN não encontrado nos Secrets do Colab (ícone da chave 🔑). Adicione 'HF_TOKEN' para acessar modelos restritos.")

DEFAULT_MODEL = "meta-llama/Llama-3.2-3B-Instruct"

def load_generation_pipeline(model_name: str, use_4bit: bool, use_8bit: bool):
    """Carrega o modelo de geração de texto com quantização opcional."""
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, trust_remote_code=True, token=HF_TOKEN)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if (use_4bit or use_8bit) and not torch.cuda.is_available():
        raise RuntimeError("Quantização 4/8-bit requer CUDA disponível no sistema.")

    if use_4bit:
        qconfig = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=qconfig,
            device_map="auto",
            trust_remote_code=True,
            token=HF_TOKEN,
        )
    elif use_8bit:
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            load_in_8bit=True,
            device_map="auto",
            trust_remote_code=True,
            token=HF_TOKEN,
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", trust_remote_code=True, token=HF_TOKEN)

    return pipeline(
        task="text-generation",
        model=model,
        tokenizer=tokenizer,
        return_full_text=False,
    )

# Avaliação Questões Objetivas

In [ ]:
#@title Carregar avaliação
def format_question_prompt(item: dict) -> str:
    """Monta prompt para questão de múltipla escolha."""
    choices = item["choices"]
    labels = choices["label"]
    texts = choices["text"]
    choices_text = "\n".join(f"{label}. {text}" for label, text in zip(labels, texts))

    prompt = (
        f"Questão {item['question_number']} (prova {item['exam_id']}):\n"
        f"{item['question']}\n\n"
        f"{choices_text}\n\n"
        "Responda apenas com a letra (A, B, C ou D). Não escreva mais nada.\n"
        "Resposta:"
    )
    print(prompt)
    return prompt

def extract_letter_answer(output: str, valid_labels: set) -> str | None:
    """Extrai a letra da resposta gerada pelo modelo."""
    normalized = output.strip().upper()
    if not normalized:
        return None

    letter_match = re.search(r"\b([A-Z])\b", normalized)
    if letter_match and letter_match.group(1) in valid_labels:
        return letter_match.group(1)

    for char in normalized:
        if char in valid_labels:
            return char
    return None

def evaluate_model(
    model_name: str,
    start_idx: int,
    count: int,
    use_4bit: bool,
    use_8bit: bool,
    output_file: str | None = None,
):
    dataset = load_dataset("eduagarcia/oab_exams", split="train", token=HF_TOKEN)
    end_idx = start_idx + count
    indices = list(range(start_idx, min(end_idx, len(dataset))))
    subset = dataset.select(indices)

    generator = load_generation_pipeline(model_name, use_4bit, use_8bit)
    results = []
    correct = 0

    for current_idx, item in zip(indices, subset):
        prompt = format_question_prompt(item)
        try:
            generation = generator(
                prompt,
                max_new_tokens=128,
                do_sample=False,
                pad_token_id=generator.tokenizer.pad_token_id,
                eos_token_id=generator.tokenizer.eos_token_id
            )
            raw_answer = generation[0]["generated_text"].strip()
        except Exception as e:
            print(f"[❌ ERRO] Questão {item['question_number']}: {str(e)}")
            raw_answer = ""

        valid_labels = set(item["choices"]["label"])
        predicted = extract_letter_answer(raw_answer, valid_labels)
        expected = item["answerKey"].strip().upper()

        is_correct = predicted == expected
        if is_correct:
            correct += 1

        results.append(
            {
                "question_idx": current_idx + 1,
                "question_number": item["question_number"],
                "exam_id": item["exam_id"],
                "predicted": predicted,
                "expected": expected,
                "raw_answer": raw_answer,
                "is_correct": is_correct,
            }
        )

        print(
            f"[{item['question_number']} | {item['exam_id']}] esperado={expected} previsto={predicted} "
            f"acerto={'SIM' if is_correct else 'NÃO'}"
        )

        # Exibe informações detalhadas sobre a questão e resposta
        print(f"\n{'='*80}")
        print(f"QUESTÃO #{item['question_number']} | PROVA: {item['exam_id']}")
        print(f"{'='*80}")
        print(f"\nPERGUNTA:\n{item['question']}")

        choices = item["choices"]
        print(f"\nALTERNATIVAS:")
        for label, text in zip(choices["label"], choices["text"]):
            print(f"  {label}. {text}")

        print(f"\nRESPOSTA BRUTA DO MODELO:\n{raw_answer if raw_answer else '[VAZIA - O modelo não gerou resposta]'}")
        print(f"\nRESULTADO:")
        print(f"  Resposta esperada: {expected}")
        print(f"  Resposta prevista: {predicted if predicted else '[NÃO EXTRAÍDA]'}")
        status = '✓ CORRETO' if is_correct else ('⚠️ VAZIA' if predicted is None else '✗ INCORRETO')
        print(f"  Status: {status}")
        print(f"{'='*80}")
        print(f"[{item['question_number']} | {item['exam_id']}] esperado={expected} previsto={predicted} acerto={'SIM' if is_correct else 'NÃO'}\n")

    accuracy = correct / len(results) if results else 0.0
    summary = {
        "model_name": model_name,
        "start_idx": start_idx,
        "count": len(results),
        "correct": correct,
        "accuracy": accuracy,
        "use_4bit": use_4bit,
        "use_8bit": use_8bit,
    }

    print("\nResumo:")
    print(f"  Questões avaliadas: {len(results)}")
    print(f"  Acertos: {correct}")
    print(f"  Acurácia: {accuracy:.2%}")

    if output_file is None:
        safe_name = model_name.replace("/", "_").replace("-", "_")
        output_file = f"{safe_name}_results.json"

    with open(output_file, "w", encoding="utf-8") as fh:
        json.dump({"summary": summary, "results": results}, fh, ensure_ascii=False, indent=2)
    print(f"Resultados salvos em: {output_file}")

    return summary, results

In [ ]:
#@title Interface de Avaliação

selected_model = "meta-llama/Llama-3.2-3B-Instruct" #@param ["google/gemma-2-2b-it", "meta-llama/Llama-3.2-3B-Instruct", "Qwen/Qwen2.5-3B-Instruct"]
quantization = "None" #@param ["None", "4-bit", "8-bit"]
start_index = 861 #@param {type:"integer"}
number_of_questions = 123 #@param {type:"integer"}

# Mapeamento da interface para os parâmetros da função
use_4bit = quantization == "4-bit"
use_8bit = quantization == "8-bit"

# Execução
summary, results = evaluate_model(
    model_name=selected_model,
    start_idx=start_index,
    count=number_of_questions,
    use_4bit=use_4bit,
    use_8bit=use_8bit
)

# Avaliação Questões Abertas

In [ ]:
#@title Carregar questões e guidelines
import requests
import json
import os
import re


# URL do dataset OAB-Bench (Questões Abertas/Discursivas)
DATASET_URL = "https://raw.githubusercontent.com/maritaca-ai/oab-bench/main/data/oab_bench/question.jsonl"
FILENAME = "oab_questions.jsonl"

def download_dataset(url, filename):
    if not os.path.exists(filename):
        print(f"Baixando {filename}...")
        response = requests.get(url)
        with open(filename, 'wb') as f:
            f.write(response.content)
        print("Download concluído.")
    else:
        print("Dataset já existe localmente.")

download_dataset(DATASET_URL, FILENAME)

def load_jsonl(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

open_questions = load_jsonl(FILENAME)
print(f"Total de questões carregadas: {len(open_questions)}")

# URL do Guia de Respostas (Guidelines)
GUIDELINES_URL = "https://raw.githubusercontent.com/maritaca-ai/oab-bench/refs/heads/main/data/oab_bench/reference_answer/guidelines.jsonl"
GUIDELINES_FILE = "oab_guidelines.jsonl"

download_dataset(GUIDELINES_URL, GUIDELINES_FILE)

def load_guidelines(filename):
    guidelines = {}
    with open(filename, 'r', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)
            guidelines[data['question_id']] = data['choices'][0]['turns']
    return guidelines

all_guidelines = load_guidelines(GUIDELINES_FILE)
print(f"Diretrizes de correção carregadas: {len(all_guidelines)}")

In [ ]:
#@title Carregar avaliação
import re
import torch
import json
from bert_score import score
from sentence_transformers import CrossEncoder

# Inicializa o Cross-Encoder para NLI
nli_model = CrossEncoder('cross-encoder/nli-deberta-v3-base', device='cuda' if torch.cuda.is_available() else 'cpu')

#nli_model = CrossEncoder('unicamp-dl/ptt5-base-portuguese-vocab', device='cuda' if torch.cuda.is_available() else 'cpu')

def get_llm_response(prompt, model_pipeline, max_tokens=256):
    """Helper para obter respostas do modelo de forma limpa."""
    try:
        generation = model_pipeline(
            prompt,
            max_new_tokens=max_tokens,
            do_sample=False,
            pad_token_id=model_pipeline.tokenizer.pad_token_id,
            eos_token_id=model_pipeline.tokenizer.eos_token_id
        )
        return generation[0]['generated_text'].strip()
    except Exception as e:
        print(f"[ERRO LLM]: {e}")
        return ""

def decompose_guidelines(reference_text, model_pipeline):
    """CAMADA 1: Decomposição em Átomos de Conhecimento (Rubricas) via LLM."""
    prompt = f"""### System:
Você é um avaliador jurídico sênior. Sua tarefa é extrair do gabarito oficial uma lista de 'rubricas' (afirmações técnicas curtas e independentes) que são obrigatórias na resposta.

Gabarito: {reference_text}

### Lista de Rubricas (uma por linha, sem numeração):
"""
    output = get_llm_response(prompt, model_pipeline, max_tokens=300)
    return [line.strip('- *') for line in output.split('\n') if len(line.strip()) > 10]

def verify_nli_layer_cross_encoder(model_response, rubric):
    """CAMADA 2: Validação Lógica NLI usando Cross-Encoder.
    Labels do modelo: 0: contradiction, 1: entailment, 2: neutral"""
    scores = nli_model.predict([(rubric, model_response)])
    label_id = scores.argmax()

    if label_id == 1: # Entailment
        return 1.0
    elif label_id == 0: # Contradiction
        return -1.5
    else: # Neutral
        return 0.0

def calculate_legal_score(model_text, reference_text, weight, model_pipeline):
    """Pipeline 'The Legal Scorer': Decomposição (LLM) + NLI (Cross-Encoder) + BERTScore."""
    if not model_text or not reference_text:
        return 0.0

    print(f"\n--- [EXECUTANDO THE LEGAL SCORER (NLI: Cross-Encoder)] ---")

    # 1. Decomposição (LLM)
    rubrics = decompose_guidelines(reference_text, model_pipeline)
    print(f"[CAMADA 1] Gabarito decomposto em {len(rubrics)} rubricas.")

    # 2. NLI (Cross-Encoder)
    nli_sum = 0
    for r in rubrics:
        status = verify_nli_layer_cross_encoder(model_text, r)
        nli_sum += status
        label = "✓" if status > 0 else ("✗" if status < 0 else "-")
        print(f"  {label} Rubrica: {r[:70]}...")

    nli_factor = max(0, nli_sum / len(rubrics)) if rubrics else 0

    # 3. BERTScore
    try:

#        P, R, F1 = score([model_text], [reference_text], model_type="neuralmind/bert-base-portuguese-cased",lang='pt',device='cuda' if torch.cuda.is_available() else 'cpu')
        P, R, F1 = score([model_text], [reference_text], lang='pt', device='cuda' if torch.cuda.is_available() else 'cpu')
        semantic_coherence = F1.mean().item()
        print(f"[CAMADA 3] BERTScore (Coerência): {semantic_coherence:.4f}")
    except:
        semantic_coherence = 0.0

    # Pesos Sugeridos: 70% Lógica (NLI), 30% Semântica (BERTScore)
    final_score = round(weight * (nli_factor * 0.7 + semantic_coherence * 0.3), 2)
    print(f"[RESULTADO]: {final_score} / {weight}")
    return min(final_score, weight)

def run_comprehensive_evaluation(model_pipeline, questions, guidelines_dict, limit=2):
    results = []
    custom_system_prompt = "Você é um bacharel em direito realizando a segunda fase da prova da OAB (FGV). Responda de forma técnica, direta e fundamentada."

    for i, item in enumerate(questions[:limit]):
        q_id = item['question_id']
        statement = item['statement']
        turns = item['turns']
        weights = item['values']

        print(f"\n{'='*80}\nPROCESSO: {q_id}\n{'='*80}")

        model_responses = []
        scores = []
        ref_turns = guidelines_dict.get(q_id, [""] * len(turns))

        for idx, turn_text in enumerate(turns):
            if len(turns) == 1 and turn_text == "":
                prompt = f"### System:\n{custom_system_prompt}\n\n### Enunciado:\n{statement}\n\n### Instrução:\nEscreva a Peça Profissional completa baseada no enunciado acima em no máx. 600 tokens.\n\n### Resposta:"
                max_tokens = 600
            else:
                prompt = f"### System:\n{custom_system_prompt}\n\n### Enunciado:\n{statement}\n\n### Pergunta Item {idx + 1}:\n{turn_text}\n\n### Instrução:\nConsiderando o enunciado, responda ao Item {idx+1} de forma direta, fundamentando tecnicamente com base legal em no max 256 tokens. \n\n### Resposta:"
                max_tokens = 256

            print(f"--- [GERANDO RESPOSTA (ITEM {idx+1})] ---")
            response_text = get_llm_response(prompt, model_pipeline, max_tokens)
            model_responses.append(response_text)

            ref_text = ref_turns[idx] if idx < len(ref_turns) else ""
            val = weights[idx] if idx < len(weights) else 0.0

            current_score = calculate_legal_score(response_text, ref_text, val, model_pipeline)
            scores.append(current_score)

        results.append({
            "question_id": q_id,
            "category": item['category'],
            "model_responses": model_responses,
            "scores": scores,
            "total_grade": sum(scores),
            "max_possible": sum(weights)
        })

    return results

### Interface de Avaliação

In [ ]:

selected_model_open = "Qwen/Qwen2.5-3B-Instruct" #@param ["google/gemma-2-2b-it", "meta-llama/Llama-3.2-3B-Instruct", "Qwen/Qwen2.5-3B-Instruct"]
quantization_open = "4-bit" #@param ["None", "4-bit", "8-bit"]
start_index_open = 82 #@param {type:"integer"}
number_of_questions_open = 12 #@param {type:"integer"}

# Mapeamento de quantização
use_4bit_open = quantization_open == "4-bit"
use_8bit_open = quantization_open == "8-bit"

# 1. Carregar Pipeline
print(f"Carregando modelo: {selected_model_open}...")
generator_open = load_generation_pipeline(
    model_name=selected_model_open,
    use_4bit=use_4bit_open,
    use_8bit=use_8bit_open
)

# 2. Executar Avaliação Integrada (Geração + Nota)
if 'open_questions' in locals() and 'all_guidelines' in locals():
    # Seleciona o intervalo desejado
    subset_questions = open_questions[start_index_open : start_index_open + number_of_questions_open]

    print(f"Iniciando geração e correção de {len(subset_questions)} questões...")
    # Agora utilizando a função que integra o calculate_legal_score
    report_results = run_comprehensive_evaluation(
        model_pipeline=generator_open,
        questions=subset_questions,
        guidelines_dict=all_guidelines,
        limit=number_of_questions_open
    )

    # 3. Salvar Resultados Detalhados
    # Gera um nome de arquivo seguro baseado no modelo selecionado
    safe_model_name = selected_model_open.replace("/", "_").replace("-", "_")
    output_filename = f"relatorio_final_{safe_model_name}.json"

    with open(output_filename, "w", encoding="utf-8") as f:
        json.dump(report_results, f, indent=4, ensure_ascii=False)

    print(f"\n✅ Avaliação e Correção concluídas!")
    print(f"📂 Relatório com notas salvo em: {output_filename}")

    # Exibição rápida no console
    for r in report_results:
        print(f"ID: {r['question_id']} | Nota: {r['total_grade']}/{r['max_possible']}")
else:
    print("❌ Erro: Verifique se o dataset e os guidelines foram carregados.")